# 03 · Resume a run after a disconnect (one click)
Select **Run All**. Nothing to edit unless you want a different run.

It mounts Drive, loads the newest code zip, copies the run's training file to local disk, rebuilds the config
**from the run's own saved `config.json`** (so settings can't drift between sessions), shows where the run stands,
and continues from `latest.pt`. Use it for any run, e.g. `A_qwen_dense_full`.

In [ ]:
RUN = 'A_qwen_dense_full'      # run folder under MyDrive/cs781/runs/ to continue
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
# ---- paths + locate the code (lm_lib/ and models/) ----
# Put the code on Drive in ONE of these ways:
#   (a) upload the folders lm_lib/ and models/ (+ hf_token.txt) into MyDrive/cs781_capstone/   [or any folder]
#   (b) upload cs781_code.zip (made by make_code_zip.py) anywhere in MyDrive -> it is unzipped automatically
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, zipfile, importlib, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must precede CUDA init
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/cs781'            # data + runs live here
DRIVE_DATA = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
LOCAL_DATA = '/content/data'

def find_code_dir():
    levels = ['', '*/', '*/*/', '*/*/*/']
    # 1) a folder on Drive containing lm_lib/ (newest wins if there are several copies)
    hits = [h for lv in levels for h in glob.glob(f'{MYDRIVE}/{lv}lm_lib/trainer.py')]
    if hits:
        return os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
    # 2) the newest cs781_code*.zip on Drive -> always re-extracted fresh (so an updated zip replaces old code)
    zips = [z for lv in levels for z in glob.glob(f'{MYDRIVE}/{lv}cs781_code*.zip')]
    if zips:
        z = max(zips, key=os.path.getmtime)
        shutil.rmtree('/content/cs781_code', ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/cs781_code')
        print('unzipped', z, '| zip modified', time.ctime(os.path.getmtime(z)))
        return '/content/cs781_code'
    raise FileNotFoundError('Could not find lm_lib/ on Drive. Upload the lm_lib/ and models/ folders '
                            '(e.g. to MyDrive/cs781_capstone/) or upload cs781_code.zip to MyDrive, then re-run.')

CODE_DIR = find_code_dir()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
importlib.invalidate_caches()
# if lm_lib/models were imported earlier in this kernel, drop them so the fresh code is used
for _m in [m for m in list(sys.modules) if m == 'lm_lib' or m.startswith(('lm_lib.', 'models.')) or m == 'models']:
    del sys.modules[_m]
print('CODE_DIR =', CODE_DIR, '| contents:', sorted(os.listdir(CODE_DIR)))
os.makedirs(DRIVE_DATA, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
import json
saved = json.load(open(f'{RUNS_DIR}/{RUN}/config.json'))
TRAIN_BIN = saved.get('train_bin', 'train.bin')
if saved['arch'].startswith('C_gdn_hybrid'):
    !pip -q install flash-linear-attention
print('run', RUN, '| arch', saved['arch'], '| train_bin', TRAIN_BIN)

In [ ]:
# copy tokenised data from Drive to fast local disk (once per runtime); the training file is TRAIN_BIN if set
import shutil, os
os.makedirs(LOCAL_DATA, exist_ok=True)
_train_files = [globals()['TRAIN_BIN']] if 'TRAIN_BIN' in globals() else []
for f in _train_files + ['val.bin', 'tokenizer.json', 'val_prompts.json', 'val_refs.json', 'val_srcs.json', 'test.json']:
    src, dst = f'{DRIVE_DATA}/{f}', f'{LOCAL_DATA}/{f}'
    if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src)):
        shutil.copyfile(src, dst)
!ls -lh {LOCAL_DATA}

In [ ]:
import pandas as pd
from lm_lib.config import TrainConfig
from lm_lib.trainer import train
cfg = TrainConfig.from_dict(saved)
cfg.data_dir, cfg.out_dir = LOCAL_DATA, RUNS_DIR
ptr = f'{RUNS_DIR}/{RUN}/latest.json'
print('latest checkpoint:', open(ptr).read() if os.path.exists(ptr) else 'none (fresh start)')
log = pd.read_csv(f'{RUNS_DIR}/{RUN}/log.csv')
v = log.dropna(subset=['val_loss']).drop_duplicates('step', keep='last')
print(v[['step', 'tokens', 'val_loss', 'val_tail_loss']].tail(5).to_string(index=False))
print(f"recent throughput {log.tok_per_s.tail(50).mean()/1e3:.1f}k tok/s, MFU {log.mfu.tail(50).mean()*100:.1f}%")

In [ ]:
result = train(cfg)     # resumes from latest.pt; after another disconnect just Run All again
result